# AI Production Studio — worker Colab personnel

Ce notebook exécute **une demande exportée manuellement** par votre studio. Il n’ouvre aucun serveur, tunnel ou port public et ne contacte jamais l’API de votre studio. Aucun abonnement ni GPU loué n’est souscrit par le code.

Les modèles ci-dessous sont des **candidats non benchmarkés sur votre session**. Image : SDXL ; vidéo : Wan 2.1 1.3B ; voix FR/AR : MMS-TTS (licence non commerciale). Vérifiez les licences sur les fiches officielles et les conditions de votre session Colab. Ne copiez pas de clé privée dans ce notebook. Les poids peuvent représenter plusieurs Go. La disponibilité GPU n’est pas garantie.

1. Dans le studio, préparez une demande dans Tâches puis choisissez « Exporter pour Colab ».
2. Dans Colab, sélectionnez votre session GPU existante pour image/vidéo. Exécutez les cellules dans l’ordre.
3. Téléversez uniquement le JSON de demande et approuvez la licence après lecture.
4. Téléchargez le ZIP de résultat puis réimportez-le dans la tâche du studio. Le fichier peut contenir un texte ou une voix privée : ne publiez ni le JSON, ni le ZIP, ni ce notebook avec ses sorties.
5. Contrôlez humainement qualité, identité et prononciation. Arrêtez votre session Colab quand vous avez terminé.

Un résultat importé peut être utilisé dans le storyboard et le montage. Aucun modèle n’est présenté comme validé avant cet essai réel. La cohérence des personnages, le lip-sync et le clonage ne sont pas intégrés à ce worker.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "diffusers==0.35.1", "transformers==4.55.4", "accelerate==1.10.1", "safetensors==0.6.2", "huggingface-hub==0.34.4", "scipy==1.16.1", "sentencepiece==0.2.1", "uroman==1.3.1", "imageio-ffmpeg==0.6.0"])
print("Dépendances installées avec vérification TLS conservée. PyTorch fourni par Colab n’a pas été remplacé.")

In [ ]:
from google.colab import files
import json, hashlib, time, zipfile
from pathlib import Path
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Téléversez un seul JSON de demande.")
raw = next(iter(uploaded.values()))
if len(raw) > 100000:
    raise ValueError("Demande trop volumineuse.")
request = json.loads(raw)
if request.get("schema_version") != 1 or request.get("task_type") not in ("image", "video", "voice"):
    raise ValueError("Demande non prise en charge.")
print("Atelier :", request["task_type"])
print("Modèle candidat :", request["model_id"])
print("Conditions :", request["license_note"])
print("Fiche officielle :", request["license_url"])
print("Aucun calcul n’est lancé avant la cellule suivante. Le plafond du studio ne mesure pas le coût d’un abonnement Colab éventuel.")

In [ ]:
APPROUVER_LICENCE_ET_SESSION = False #@param {type:"boolean"}
LANGUE_VOIX = "fr" #@param ["fr", "ar"]
if not APPROUVER_LICENCE_ET_SESSION:
    raise RuntimeError("Lisez les conditions du modèle et de votre session, puis cochez l’approbation pour continuer.")
import torch
if request["task_type"] in ("image", "video") and not torch.cuda.is_available():
    raise RuntimeError("GPU absent. Ne lancez pas cette génération lourde sur CPU : vérifiez votre session Colab.")
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (voix seulement)")
print("PyTorch :", torch.__version__)
from huggingface_hub import model_info
allowed = {"image": "stabilityai/stable-diffusion-xl-base-1.0", "video": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers", "voice": "facebook/mms-tts-fra"}
if request["model_id"] != allowed[request["task_type"]]:
    raise ValueError("Modèle non autorisé par ce notebook.")
model_id = "facebook/mms-tts-ara" if request["task_type"] == "voice" and LANGUE_VOIX == "ar" else request["model_id"]
revision = model_info(model_id, token=False).sha
print("Révision officielle figée pour cette exécution :", revision)
seed = int(request.get("seed", 42))
prompt = str(request["prompt"])
if not prompt.strip() or len(prompt) > 6000:
    raise ValueError("Prompt vide ou trop long.")
if request["task_type"] == "voice" and len(prompt) > 2000:
    raise ValueError("Voix : 2000 caractères maximum par demande. Segmentez le texte dans le studio.")
torch.manual_seed(seed)
started = time.perf_counter()

In [ ]:
task = request["task_type"]
if task == "image":
    from diffusers import StableDiffusionXLPipeline
    pipe = StableDiffusionXLPipeline.from_pretrained(model_id, revision=revision, torch_dtype=torch.float16, use_safetensors=True, token=False)
    pipe.enable_model_cpu_offload()
    loaded = time.perf_counter()
    image = pipe(prompt=prompt, width=512, height=512, num_inference_steps=20, generator=torch.Generator(device="cuda").manual_seed(seed)).images[0]
    output = Path("output.png")
    image.save(output)
    parameters = {"width":512,"height":512,"steps":20,"seed":seed}
elif task == "video":
    from diffusers import AutoencoderKLWan, WanPipeline
    from diffusers.utils import export_to_video
    vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", revision=revision, torch_dtype=torch.float32, use_safetensors=True, token=False)
    pipe = WanPipeline.from_pretrained(model_id, revision=revision, vae=vae, torch_dtype=torch.float16, use_safetensors=True, token=False)
    pipe.enable_model_cpu_offload()
    loaded = time.perf_counter()
    frames = pipe(prompt=prompt, height=480, width=832, num_frames=81, num_inference_steps=30, guidance_scale=5.0, generator=torch.Generator(device="cuda").manual_seed(seed)).frames[0]
    output = Path("output.mp4")
    export_to_video(frames, str(output), fps=16)
    parameters = {"width":832,"height":480,"frames":81,"fps":16,"steps":30,"seed":seed}
else:
    from transformers import AutoTokenizer, VitsModel
    import scipy.io.wavfile
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, token=False)
    model = VitsModel.from_pretrained(model_id, revision=revision, token=False, use_safetensors=True)
    inputs = tokenizer(prompt, return_tensors="pt")
    loaded = time.perf_counter()
    with torch.no_grad():
        audio = model(**inputs).waveform[0].cpu().numpy()
    output = Path("output.wav")
    scipy.io.wavfile.write(output, rate=model.config.sampling_rate, data=audio)
    parameters = {"language":LANGUE_VOIX,"sampling_rate":model.config.sampling_rate,"seed":seed,"max_characters":2000}
finished = time.perf_counter()
if not output.is_file() or not output.stat().st_size:
    raise RuntimeError("Le moteur n’a pas produit de fichier.")
print("Temps de chargement :", round(loaded-started,2), "s")
print("Temps de génération :", round(finished-loaded,2), "s")
print("Résultat produit. Vérifiez-le visuellement ou à l’écoute avant de l’utiliser.")

In [ ]:
result = {
    "schema_version":1, "job_id":request["job_id"], "run_id":request["run_id"], "result_token":request["result_token"],
    "filename":output.name, "checksum":hashlib.sha256(output.read_bytes()).hexdigest(),
    "model_id":model_id, "model_revision":revision, "runtime_sec":round(finished-started,3),
    "load_sec":round(loaded-started,3), "generation_sec":round(finished-loaded,3), "parameters":parameters,
    "gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "torch_version":torch.__version__,
    "license_url":"https://huggingface.co/"+model_id,
    "warnings":["Modèle candidat : validation humaine et benchmark requis.","Coût éventuel de la session Colab non mesuré par le studio."]
}
archive = Path("resultat-studio.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.writestr("result.json",json.dumps(result,ensure_ascii=False,indent=2))
    z.write(output,output.name)
files.download(str(archive))
print("Dans votre studio : Tâches → Importer le résultat Colab. Ne publiez pas ce ZIP. Pensez à arrêter votre session.")

## En cas de panne

- GPU indisponible / mémoire insuffisante : ne lancez pas de location payante automatiquement. Arrêtez l’essai et relevez le message d’erreur sans partager de secret.
- Modèle inaccessible ou licence modifiée : consultez sa fiche officielle. N’utilisez ni miroir non vérifié, ni désactivation TLS, ni fichier pickle inconnu pour contourner l’erreur.
- Image noire, voix mal prononcée ou vidéo incohérente : le modèle n’est pas validé. Conservez le rapport de temps et le résultat pour comparer ; ne supposez pas une garantie de qualité.
- Le notebook est livré et son code est vérifié syntaxiquement, mais **n’a pas été exécuté sur GPU depuis l’environnement Codex**. Les trois moteurs restent expérimentaux tant que votre session ne les a pas exécutés avec succès.
